# Transformer all tasks — robustness results from script

This notebook reads CSV artifacts generated by `transformer_robustness.py`.

The Transformer robustness training/evaluation itself is kept in a standalone Python script because the original notebook had kernel instability. This result notebook stays lightweight: it loads the generated CSV files and presents the same three sections as the MLP/CNN/LSTM robustness notebooks:

1. clean-train robustness results
2. augmented-train robustness results
3. clean-vs-aug score comparison

For the main fair comparison, use the manually pasted `FINAL_TUNED_CFGS` from `transformer_hyperparameter_tuning.ipynb` inside the Python script before running it.


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

The generating script is:

```bash
mkdir notebooks/logs/transformer_robustness
python notebooks/transformer_robustness.py --output-dir notebooks/logs/transformer_robustness
```

or, if you run it from another folder:

```bash
mkdir notebooks/logs/transformer_robustness
python transformer_robustness.py --project-root /path/to/CS7643-Project --output-dir /path/to/CS7643-Project/notebooks/logs/transformer_robustness
```

This notebook reads the saved CSV outputs and presents them in the same order as the MLP/CNN/LSTM robustness notebooks.


In [2]:
# Resolve paths assuming this notebook is run from project_root/notebooks
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
LOG_DIR = PROJECT_ROOT / "notebooks" / "logs" / "transformer_robustness"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("LOG_DIR      =", LOG_DIR)

if not LOG_DIR.exists():
    raise FileNotFoundError(f"Could not find logs directory: {LOG_DIR}")


def find_csv_by_name(log_dir: Path, filename: str) -> Path:
    matches = sorted(log_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Missing required CSV: {filename} under {log_dir}")

    # Prefer a top-level file if it exists, otherwise require uniqueness.
    top_level = [p for p in matches if p.parent == log_dir]
    if len(top_level) == 1:
        return top_level[0]
    if len(matches) == 1:
        return matches[0]

    rels = "\n".join(str(p.relative_to(log_dir)) for p in matches)
    raise ValueError(
        f"Found multiple candidates for {filename}. Keep one canonical file or edit selector.\n{rels}"
    )


required_files = {
    "tuned_hparams": "tuned_hparams.csv",
    "results_clean": "results_clean.csv",
    "results_aug": "results_aug.csv",
    "cmp_delta_score": "cmp_delta_score.csv",
    "cmp_delta_score_pivot": "cmp_delta_score_pivot.csv",
    "best_metrics_summary": "best_metrics_summary.csv",
}

csv_paths = {key: find_csv_by_name(LOG_DIR, fname) for key, fname in required_files.items()}
for key, path in csv_paths.items():
    print(f"{key:>20} : {path.relative_to(LOG_DIR)}")

tuned_hparams = pd.read_csv(csv_paths["tuned_hparams"])
results_clean = pd.read_csv(csv_paths["results_clean"])
results_aug = pd.read_csv(csv_paths["results_aug"])
cmp_delta_score = pd.read_csv(csv_paths["cmp_delta_score"])
cmp_delta_score_pivot = pd.read_csv(csv_paths["cmp_delta_score_pivot"])
best_metrics_summary = pd.read_csv(csv_paths["best_metrics_summary"])


PROJECT_ROOT = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project
LOG_DIR      = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/transformer_robustness
       tuned_hparams : tuned_hparams.csv
       results_clean : results_clean.csv
         results_aug : results_aug.csv
     cmp_delta_score : cmp_delta_score.csv
cmp_delta_score_pivot : cmp_delta_score_pivot.csv
best_metrics_summary : best_metrics_summary.csv


In [3]:
# Quick structural check: the transformer result notebook should mirror the other robustness notebooks.
print("tuned_hparams shape      :", tuned_hparams.shape)
print("results_clean shape      :", results_clean.shape)
print("results_aug shape        :", results_aug.shape)
print("cmp_delta_score shape    :", cmp_delta_score.shape)
print("pivot shape              :", cmp_delta_score_pivot.shape)
print("best_metrics_summary     :", best_metrics_summary.shape)

required_cmp_cols = {
    "scenario",
    "model",
    "score_clean_train",
    "score_aug_train",
    "delta_score",
}
missing = required_cmp_cols - set(cmp_delta_score.columns)
if missing:
    raise ValueError(f"cmp_delta_score.csv is missing columns: {missing}")

print("\nFine-tuned configs used by the script:")
display(tuned_hparams)

print("\nBest clean/aug training metrics:")
display(best_metrics_summary)


tuned_hparams shape      : (6, 18)
results_clean shape      : (108, 12)
results_aug shape        : (108, 12)
cmp_delta_score shape    : (108, 5)
pivot shape              : (18, 7)
best_metrics_summary     : (12, 14)

Fine-tuned configs used by the script:


,family,task,lr,weight_decay,max_epochs,patience,print_every,batch_size,val_batch_size,grad_clip_norm,arch_d_model,arch_nhead,arch_num_layers,arch_dim_feedforward,arch_dropout,arch_num_heads,arch_num_sab_layers,arch_num_seed_vectors
0,set,coordinate,0.0010,0.0005,50,10,5,256,512,1.0,128,NaN,NaN,256,0.1,4.0,2.0,1.0
1,standard,coordinate,0.0010,0.0001,50,10,5,256,512,1.0,128,4.0,2.0,256,0.1,NaN,NaN,NaN
2,set,joint,0.0005,0.0005,80,15,5,256,512,1.0,128,NaN,NaN,256,0.1,4.0,2.0,1.0
3,standard,joint,0.0010,0.0005,50,10,5,256,512,1.0,128,4.0,2.0,256,0.1,NaN,NaN,NaN
4,set,multitask,0.0005,0.0005,80,15,5,256,512,1.0,128,NaN,NaN,256,0.1,4.0,2.0,1.0
5,standard,multitask,0.0010,0.0001,50,10,5,256,512,1.0,128,4.0,2.0,256,0.1,NaN,NaN,NaN



Best clean/aug training metrics:


,model,train_regime,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,transformer_joint,clean_train,48,48.0,0.011816,0.369757,0.943294,0.943294,1.0000,0.943294,NaN,NaN,NaN,NaN
1,set_transformer_joint,clean_train,41,41.0,0.019608,0.365364,0.938794,0.938794,0.9991,0.938794,NaN,NaN,NaN,NaN
2,transformer_multitask,clean_train,21,21.0,0.062508,0.240464,0.954996,0.954996,0.9991,0.954996,NaN,NaN,NaN,NaN
3,set_transformer_multitask,clean_train,28,28.0,0.086580,0.301406,0.934293,0.934293,0.9955,0.936094,NaN,NaN,NaN,NaN
4,transformer_coordinate,clean_train,40,40.0,0.010951,0.012482,-9.995122,NaN,NaN,NaN,0.113662,0.154118,9.995122,13.323050
5,set_transformer_coordinate,clean_train,50,50.0,0.009856,0.010813,-10.131620,NaN,NaN,NaN,0.113325,0.146442,10.131620,13.051028
6,transformer_joint,aug_train,11,11.0,0.103443,0.222953,0.945995,0.945995,0.9982,0.945995,NaN,NaN,NaN,NaN
7,set_transformer_joint,aug_train,12,12.0,0.126912,0.223655,0.949595,0.949595,0.9991,0.949595,NaN,NaN,NaN,NaN
8,transformer_multitask,aug_train,20,20.0,0.074884,0.189962,0.958596,0.958596,1.0000,0.958596,NaN,NaN,NaN,NaN
9,set_transformer_multitask,aug_train,15,15.0,0.133876,0.277100,0.938794,0.938794,0.9964,0.940594,NaN,NaN,NaN,NaN


## 1) Clean-train robustness results

This is the transformer-side equivalent of the MLP notebook cell that builds `results_clean`.

These rows answer:

- how each clean-trained transformer model performs on every degraded-condition scenario
- which exact scenario/model pairs are being compared later


In [4]:
display(results_clean)

print("Models in results_clean:")
print(sorted(results_clean["model"].unique()))

print("\nScenarios in results_clean:")
print(results_clean["scenario"].tolist() if results_clean["scenario"].nunique() <= 25 else sorted(results_clean["scenario"].unique()))

,scenario,score,joint_accuracy,building_accuracy,floor_accuracy,eval_loss,model,train_regime,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,clean,0.943294,0.943294,1.000000,0.943294,0.369757,transformer_joint,clean_train,NaN,NaN,NaN,NaN
1,dropout_0.05,0.935194,0.935194,1.000000,0.935194,0.410916,transformer_joint,clean_train,NaN,NaN,NaN,NaN
2,dropout_0.10,0.929793,0.929793,0.998200,0.929793,0.464283,transformer_joint,clean_train,NaN,NaN,NaN,NaN
3,dropout_0.15,0.924392,0.924392,0.998200,0.924392,0.511468,transformer_joint,clean_train,NaN,NaN,NaN,NaN
4,dropout_0.20,0.918092,0.918092,0.994599,0.918992,0.549671,transformer_joint,clean_train,NaN,NaN,NaN,NaN
5,dropout_0.25,0.898290,0.898290,0.992799,0.899190,0.616090,transformer_joint,clean_train,NaN,NaN,NaN,NaN
6,dropout_0.30,0.895590,0.895590,0.997300,0.895590,0.718300,transformer_joint,clean_train,NaN,NaN,NaN,NaN
7,dropout_0.35,0.867687,0.867687,0.992799,0.868587,0.912811,transformer_joint,clean_train,NaN,NaN,NaN,NaN
8,dropout_0.40,0.834383,0.834383,0.992799,0.836184,1.078135,transformer_joint,clean_train,NaN,NaN,NaN,NaN
9,dropout_0.45,0.824482,0.824482,0.981998,0.828983,1.243149,transformer_joint,clean_train,NaN,NaN,NaN,NaN


Models in results_clean:
['set_transformer_coordinate', 'set_transformer_joint', 'set_transformer_multitask', 'transformer_coordinate', 'transformer_joint', 'transformer_multitask']

Scenarios in results_clean:
['clean', 'dropout_0.05', 'dropout_0.10', 'dropout_0.15', 'dropout_0.20', 'dropout_0.25', 'dropout_0.30', 'dropout_0.35', 'dropout_0.40', 'dropout_0.45', 'dropout_0.50', 'dropout_0.60', 'drop0.25_bias3_noise1', 'drop0.35_bias4_noise2', 'drop0.40_bias5_noise3', 'drop0.15_bias6_noise0', 'bias5_only', 'noise3_only', 'clean', 'dropout_0.05', 'dropout_0.10', 'dropout_0.15', 'dropout_0.20', 'dropout_0.25', 'dropout_0.30', 'dropout_0.35', 'dropout_0.40', 'dropout_0.45', 'dropout_0.50', 'dropout_0.60', 'drop0.25_bias3_noise1', 'drop0.35_bias4_noise2', 'drop0.40_bias5_noise3', 'drop0.15_bias6_noise0', 'bias5_only', 'noise3_only', 'clean', 'dropout_0.05', 'dropout_0.10', 'dropout_0.15', 'dropout_0.20', 'dropout_0.25', 'dropout_0.30', 'dropout_0.35', 'dropout_0.40', 'dropout_0.45', 'dropou

## 2) Augmented-train robustness results

This is the transformer-side equivalent of the MLP notebook cell that builds `results_aug`.

These rows answer:

- how each augmented-trained transformer model performs on the same robustness grid
- whether augmented training helps or hurts under each scenario


In [5]:
display(results_aug)

print("Models in results_aug:")
print(sorted(results_aug["model"].unique()))

print("\nScenario alignment check (clean vs aug):")
print(sorted(results_clean["scenario"].unique()) == sorted(results_aug["scenario"].unique()))

print("Model alignment check (clean vs aug):")
print(sorted(results_clean["model"].unique()) == sorted(results_aug["model"].unique()))

,scenario,score,joint_accuracy,building_accuracy,floor_accuracy,eval_loss,model,train_regime,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,clean,0.945995,0.945995,0.998200,0.945995,0.222953,transformer_joint,aug_train,NaN,NaN,NaN,NaN
1,dropout_0.05,0.942394,0.942394,0.998200,0.942394,0.224650,transformer_joint,aug_train,NaN,NaN,NaN,NaN
2,dropout_0.10,0.936094,0.936094,0.998200,0.936094,0.253305,transformer_joint,aug_train,NaN,NaN,NaN,NaN
3,dropout_0.15,0.927993,0.927993,0.998200,0.927993,0.274289,transformer_joint,aug_train,NaN,NaN,NaN,NaN
4,dropout_0.20,0.929793,0.929793,0.999100,0.929793,0.271903,transformer_joint,aug_train,NaN,NaN,NaN,NaN
5,dropout_0.25,0.919892,0.919892,0.997300,0.919892,0.287677,transformer_joint,aug_train,NaN,NaN,NaN,NaN
6,dropout_0.30,0.912691,0.912691,0.998200,0.912691,0.339764,transformer_joint,aug_train,NaN,NaN,NaN,NaN
7,dropout_0.35,0.891989,0.891989,0.995500,0.891989,0.421697,transformer_joint,aug_train,NaN,NaN,NaN,NaN
8,dropout_0.40,0.887489,0.887489,0.998200,0.887489,0.429453,transformer_joint,aug_train,NaN,NaN,NaN,NaN
9,dropout_0.45,0.874888,0.874888,0.997300,0.874888,0.513506,transformer_joint,aug_train,NaN,NaN,NaN,NaN


Models in results_aug:
['set_transformer_coordinate', 'set_transformer_joint', 'set_transformer_multitask', 'transformer_coordinate', 'transformer_joint', 'transformer_multitask']

Scenario alignment check (clean vs aug):
True
Model alignment check (clean vs aug):
True


## 3) Compare `score` (clean train vs aug train)

This is the transformer-side equivalent of the MLP notebook cells that create:

- `cmp`
- the pivot table of `delta_score`

Interpretation:

- `delta_score = score_aug_train - score_clean_train`
- positive means augmented training improved the score
- negative means augmented training hurt the score

For coordinate models, confirm that the stored `score` already uses the same higher-is-better convention as the MLP notebook before drawing conclusions.

In [6]:
display(cmp_delta_score)
display(cmp_delta_score_pivot)

print("Standard Transformer rows:")
display(cmp_delta_score[cmp_delta_score["model"].str.startswith("transformer_")])

print("Set Transformer rows:")
display(cmp_delta_score[cmp_delta_score["model"].str.startswith("set_transformer_")])

,scenario,model,score_clean_train,score_aug_train,delta_score
0,clean,set_transformer_coordinate,-10.131620,-13.679850,-3.548230
1,dropout_0.05,set_transformer_coordinate,-10.418535,-13.844310,-3.425776
2,dropout_0.10,set_transformer_coordinate,-10.763223,-14.139616,-3.376393
3,dropout_0.15,set_transformer_coordinate,-10.952308,-14.419107,-3.466799
4,dropout_0.20,set_transformer_coordinate,-11.297670,-14.716292,-3.418622
5,dropout_0.25,set_transformer_coordinate,-11.677921,-14.989415,-3.311494
6,dropout_0.30,set_transformer_coordinate,-12.032147,-15.512732,-3.480585
7,dropout_0.35,set_transformer_coordinate,-12.738518,-16.108176,-3.369658
8,dropout_0.40,set_transformer_coordinate,-13.539546,-16.687886,-3.148340
9,dropout_0.45,set_transformer_coordinate,-14.578847,-17.614402,-3.035555


,scenario,set_transformer_coordinate,set_transformer_joint,set_transformer_multitask,transformer_coordinate,transformer_joint,transformer_multitask
0,clean,-3.548230,0.010801,0.004500,0.813263,0.002700,0.003600
1,dropout_0.05,-3.425776,0.012601,0.015302,0.864455,0.007201,0.005401
2,dropout_0.10,-3.376393,0.014401,0.013501,1.013018,0.006301,0.014401
3,dropout_0.15,-3.466799,0.009901,0.020702,1.116641,0.003600,0.016202
4,dropout_0.20,-3.418622,0.016202,0.027003,1.163955,0.011701,0.019802
5,dropout_0.25,-3.311494,0.009001,0.029703,1.456203,0.021602,0.018902
6,dropout_0.30,-3.480585,0.030603,0.025203,1.345676,0.017102,0.005401
7,dropout_0.35,-3.369658,0.024302,0.035104,2.091756,0.024302,0.027903
8,dropout_0.40,-3.148340,0.015302,0.052205,2.707890,0.053105,0.036004
9,dropout_0.45,-3.035555,0.038704,0.035104,2.457906,0.050405,0.039604


Standard Transformer rows:


,scenario,model,score_clean_train,score_aug_train,delta_score
54,clean,transformer_coordinate,-9.995122,-9.181859,0.813263
55,dropout_0.05,transformer_coordinate,-10.182077,-9.317622,0.864455
56,dropout_0.10,transformer_coordinate,-10.564780,-9.551762,1.013018
57,dropout_0.15,transformer_coordinate,-10.830583,-9.713941,1.116641
58,dropout_0.20,transformer_coordinate,-11.251227,-10.087272,1.163955
59,dropout_0.25,transformer_coordinate,-11.496462,-10.040259,1.456203
60,dropout_0.30,transformer_coordinate,-11.829665,-10.483989,1.345676
61,dropout_0.35,transformer_coordinate,-12.902731,-10.810975,2.091756
62,dropout_0.40,transformer_coordinate,-13.887493,-11.179603,2.707890
63,dropout_0.45,transformer_coordinate,-14.764481,-12.306575,2.457906


Set Transformer rows:


,scenario,model,score_clean_train,score_aug_train,delta_score
0,clean,set_transformer_coordinate,-10.131620,-13.679850,-3.548230
1,dropout_0.05,set_transformer_coordinate,-10.418535,-13.844310,-3.425776
2,dropout_0.10,set_transformer_coordinate,-10.763223,-14.139616,-3.376393
3,dropout_0.15,set_transformer_coordinate,-10.952308,-14.419107,-3.466799
4,dropout_0.20,set_transformer_coordinate,-11.297670,-14.716292,-3.418622
5,dropout_0.25,set_transformer_coordinate,-11.677921,-14.989415,-3.311494
6,dropout_0.30,set_transformer_coordinate,-12.032147,-15.512732,-3.480585
7,dropout_0.35,set_transformer_coordinate,-12.738518,-16.108176,-3.369658
8,dropout_0.40,set_transformer_coordinate,-13.539546,-16.687886,-3.148340
9,dropout_0.45,set_transformer_coordinate,-14.578847,-17.614402,-3.035555


## Appendix — inventory of available CSV files

These cells are optional. They help when the log directory contains extra files or multiple seed-run folders.

In [7]:
csv_paths_all = sorted(LOG_DIR.rglob("*.csv"))
inventory_rows = []
for p in csv_paths_all:
    rel = p.relative_to(LOG_DIR)
    inventory_rows.append({
        "relative_path": str(rel),
        "parent": str(rel.parent),
        "filename": p.name,
        "size_bytes": p.stat().st_size,
    })

inventory_df = pd.DataFrame(inventory_rows).sort_values(["filename", "relative_path"]).reset_index(drop=True)
display(inventory_df)

,relative_path,parent,filename,size_bytes
0,best_metrics_summary.csv,.,best_metrics_summary.csv,2179
1,cmp_delta_score.csv,.,cmp_delta_score.csv,10411
2,cmp_delta_score_pivot.csv,.,cmp_delta_score_pivot.csv,2590
3,results_aug.csv,.,results_aug.csv,16606
4,results_clean.csv,.,results_clean.csv,16916
5,tuned_hparams.csv,.,tuned_hparams.csv,672
